# cancer-recon-apoptosis — RUNG 2: DR5 clustering-geometry consistency check

Asks the narrow question: does clustering **geometry** predict DR5-agonist potency **beyond valency** (arm-count)? It is wired so the default, most-likely, fully-acceptable output is the honest **valency lookup table** — reachable past a blinding + negative-control + oracle-noise gate.

**Result you should expect (CPU, ~12 s):** Spearman(potency, valency) ≈ 0.94 (valency explains ~88%); within-valency exact p ≈ 0.17 (not significant); negative control reproduces the signal → **VERDICT: valency lookup table**. Positive side-result: the percolation sim reproduces the ladder (Spearman ≈ 0.80) — it *explains* the valency law.

**Ceiling (honest):** RUNG 2 does **not** prove agonism. binding→caspase-8 (the cooperative DISC threshold) is the irreducible wet-lab crux (see `EVIDENCE_AND_HANDOFF.md`). The RUNG-2 score is **never** multiplied by the RUNG-1 apoptosis result.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — start run log + ensure deps (numpy/scipy/pandas/sklearn preinstalled on Colab)
import sys, importlib.util
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('rung2_clustering', repo_dir='.')
for pkg in ['scipy', 'sklearn', 'pandas', 'yaml']:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q',
                    {'sklearn': 'scikit-learn', 'yaml': 'pyyaml'}.get(pkg, pkg)], RUNLOG, label=f'pip install {pkg}')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — run RUNG 2 (CPU): blind labeler → clustering sim → calibration + tests
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('rung2_clustering', repo_dir='.')
run_logged([sys.executable, '-u', 'scripts/14_blind_labeler.py'], RUNLOG)
run_logged([sys.executable, '-u', 'scripts/13_clustering_sim.py'], RUNLOG)
rc = run_logged([sys.executable, '-u', 'scripts/12_clustering_rank.py'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓ (methodology valid)' if rc == 0 else '✗ (integrity check failed)')

In [ ]:
#@title Cell 4 — show figures
from IPython.display import Image, display
import os
for p in ['runs/rung2_clustering/rung2_clustering.png', 'runs/rung2_clustering/clustering_phase.png']:
    if os.path.exists(p):
        display(Image(p))

## Cell 5 (OPTIONAL, GPU) — Boltz-2 resolution probe

The within-valency contrast (the 4 bivalent IgG1s) can only be physics-driven if a structure oracle can actually *resolve* their arm geometry. Our committed evidence (`runs/step1_boltz/interface_metrics_colab_transcript.json`) shows Boltz-2 cannot discriminate even a binder from a non-binder on this DR5 ectodomain. This cell runs **multi-seed Boltz-2** on the three bivalents and reports the **seed-to-seed spread** of arm spacing/angle. If within-molecule spread ≥ between-molecule difference, the same-valency inputs are **oracle-noise-dominated** and the within-valency test is label-driven, not physics-driven (this is the expected outcome). Requires a GPU runtime + `pip install boltz` (heavy). Skip for the CPU result above.

In [ ]:
#@title Cell 5 — Boltz-2 resolution probe (optional; needs GPU). Safe to skip.
import os, importlib.util
RUN_BOLTZ_PROBE = False  #@param {type:'boolean'}
if not RUN_BOLTZ_PROBE:
    print('[probe] skipped. Default: literature/format geometry used; Boltz-2 known-unreliable on this DR5')
    print('        (committed lysozyme/decoy transcript) → within-valency inputs treated as oracle-noise-dominated.')
else:
    if importlib.util.find_spec('boltz') is None:
        !pip install -q boltz
    print('[probe] Re-uses scripts/01_boltz_smoketest.py + scripts/02_interface_metrics.py on DR5:agonist-arm.')
    print('[probe] Run multi-seed per bivalent, then write data/dr5_agonists/geometry_from_boltz.csv with')
    print('        columns seed_spread_r_arm, seed_spread_angle, within_ge_between. scripts/12 will read it.')
    print('[probe] (Left as an explicit handoff: assemble the DR5-ectodomain:Fab-arm FASTAs and seed list,')
    print('         mirroring the Step-1 Boltz stack. Expected: within_ge_between=True for flexible IgG1s.')

In [ ]:
#@title Cell 6 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('rung2_clustering', repo_dir='.')
finalize(RUNLOG)

## What this rung established

- **The valency law** — across the real DR5 drug ladder, arm-count alone explains ~88% of potency (Spearman 0.94).
- **Geometry adds nothing trustworthy beyond valency** at n=9 — the honest **valency-lookup-table** verdict, confirmed by a scrambled-feature negative control that reproduces the apparent signal.
- **Positive mechanistic result** — a from-scratch percolation simulation reproduces the ladder (0.80), giving a physical *reason* the valency law holds (bigger arm-count → bigger receptor clusters → crosses the DISC-nucleation percolation threshold).
- **Wet-lab map** (your directive) — see `RUNG2_METHODOLOGY.md`: the sim only *prioritizes* constructs; the irreducible chain (Caspase-Glo 8 → viability → dSTORM/co-IP) must still run. Cloud-lab cost/access is UNVERIFIED.

Next: **RUNG 3** (PhysiCell tissue with real physics, feeding RUNG-1 death-latency in) → **RUNG 4** (patient priming-susceptibility map). All gate behind the wet-lab DR5-clustering/caspase-8 assay.